# ST554_Project 2
Author: Huiping Zhou   
Data: 3/14/2026

# Part I-Creating a Class
In this section, we are going to create a class called `SparkDataCheck` and save it on a .py file. We will test the SparkDataCheck class on the [air](https://www4.stat.ncsu.edu/online/datasets/air.csv) dataset which read in as Spark SQL style data frames.

In [1]:
#import needed modules
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd
from pandas.api.types import is_numeric_dtype
from pyspark.sql.types import StringType

In [2]:
# Create a SparkSession named 'my_app'
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('my_app').config("spark.sql.ansi.enabled", "false").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/17 17:36:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/17 17:36:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


We read in dataset `air` using the method that created in class file. we import the my_class into this notebook first.

In [3]:
import my_class

In [5]:
#you can run this code and this will re-import that class that you have there without having to restart your kernel.
import importlib
importlib.reload(my_class)

<module 'my_class' from '/home/jupyter-hzhou13@ncsu.edu/Project2/my_class.py'>

### Reading in dataset using `from_spark` method
We use the `from_spark` method to read in the dataset and dispaly the first 5 rows using the `show()` method.

In [4]:
sql_df = my_class.SparkDataCheck.from_spark(
    spark,
    "air.csv",
    format="csv",
    #header=True,
    #inferSchema=True,
    sep=",")
sql_df.show(5)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-17 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-17 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-17 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-17 21:00:00|   2.2|       137

26/03/17 17:37:05 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hzhou13@ncsu.edu/Project2/air.csv


Based on the output above, the `from_spark` method successfully read the data correctly.

In [5]:
#check the data types of clomuns
sql_df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08.S1(CO): integer (nullable = true)
 |-- NMHC(GT): integer (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08.S2(NMHC): integer (nullable = true)
 |-- NOx(GT): integer (nullable = true)
 |-- PT08.S3(NOx): integer (nullable = true)
 |-- NO2(GT): integer (nullable = true)
 |-- PT08.S4(NO2): integer (nullable = true)
 |-- PT08.S5(O3): integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)



We will update column names that include dots (.) to avoid potential problems when working with Spark DataFrames.

In [6]:
sql_air = (sql_df
        .withColumnRenamed('PT08.S1(CO)', 'S1_CO')
        .withColumnRenamed('PT08.S2(NMHC)', 'S2_NMHC')
        .withColumnRenamed('PT08.S3(NOx)', 'S3_NOx')
        .withColumnRenamed('PT08.S4(NO2)', 'S4_NO2')
        .withColumnRenamed('PT08.S5(O3)', 'S5_O3'))
sql_air.show(5)

+---+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|_c0|     Date|               Time|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|
+---+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|  0|3/10/2004|2026-03-17 18:00:00|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-17 19:00:00|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-17 20:00:00|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-17 21:00:00|   2.2| 1376|      80|     9.2|    948|    172|  1092|    122|  1584| 1203|11.0|60.0|0.7867|
|  4|3/10/2004|2026-03-17 22:00:00|   1.6| 1272|      51|     6.5|    836|    131|  1205|    116|  1490|

26/03/17 17:38:04 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hzhou13@ncsu.edu/Project2/air.csv


Since we know that this dataset uses -200 to code missing values, we will check whether -200 appears in any column by generating basic summary statistics using the `summary()` method.

In [7]:
num_cols = ['CO(GT)', 'S1_CO', 'NMHC(GT)', 'C6H6(GT)', 'S2_NMHC', 'NOx(GT)', 'S3_NOx', 'NO2(GT)',  'S4_NO2', 'S5_O3', 'T', 'RH', 'AH']
sql_air.select(*num_cols).summary('count', 'mean', 'stddev', 'min', 'max').show()

26/03/17 17:38:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+-------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------+
|summary|            CO(GT)|             S1_CO|           NMHC(GT)|          C6H6(GT)|          S2_NMHC|          NOx(GT)|            S3_NOx|           NO2(GT)|            S4_NO2|             S5_O3|                T|               RH|                AH|
+-------+------------------+------------------+-------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------+
|  count|              9357|              9357|               9357|              9357|             9357|             9357|              9357|              9357|              9357|              9357|             9357|             9357|    

The basic summary statistics show that all numeric variables have a minimum value of -200, which is the sentinel value used in this dataset to represent missing observations. Then we will replace the coded missing values(-200) with NULL.

In [8]:
#replace the coded missing value (-200) with NULL in all numeric columns
sql_air =sql_air.na.replace({-200: None})
sql_air.show()

+---+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|_c0|     Date|               Time|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|
+---+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|  0|3/10/2004|2026-03-17 18:00:00|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-17 19:00:00|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-17 20:00:00|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-17 21:00:00|   2.2| 1376|      80|     9.2|    948|    172|  1092|    122|  1584| 1203|11.0|60.0|0.7867|
|  4|3/10/2004|2026-03-17 22:00:00|   1.6| 1272|      51|     6.5|    836|    131|  1205|    116|  1490|

26/03/17 17:42:54 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hzhou13@ncsu.edu/Project2/air.csv


Next, we will show examples of calling each method from the my_class file on this object, beginning with 
### Testing the `check_numeric_range method`   
**Case 1**: Provide two bounds (2,5)   
When two bounds are provided, the `check_numeric_range` method appends a Boolean column named `new_CO(GT)` to the returned DataFrame.
- If the value in the selected column falls within the specified range, the method returns true.
- If the value is outside the range, it returns false.
- If the value is null, the method returns null in the new column.

In [13]:
# Create a SparkDataCheck instance from the existing sql_air DataFrame
sql_air_checker = my_class.SparkDataCheck(sql_air)

#case 1:provide two bounds (2,5)
check=sql_air_checker.check_numeric_range(column = 'CO(GT)', lower = 2, upper = 5, new_column = 'new_CO(GT)')
check.select('Date', 'CO(GT)', 'new_CO(GT)').show(40)

+---------+------+----------+
|     Date|CO(GT)|new_CO(GT)|
+---------+------+----------+
|3/10/2004|   2.6|      true|
|3/10/2004|   2.0|      true|
|3/10/2004|   2.2|      true|
|3/10/2004|   2.2|      true|
|3/10/2004|   1.6|     false|
|3/10/2004|   1.2|     false|
|3/11/2004|   1.2|     false|
|3/11/2004|   1.0|     false|
|3/11/2004|   0.9|     false|
|3/11/2004|   0.6|     false|
|3/11/2004|  NULL|      NULL|
|3/11/2004|   0.7|     false|
|3/11/2004|   0.7|     false|
|3/11/2004|   1.1|     false|
|3/11/2004|   2.0|      true|
|3/11/2004|   2.2|      true|
|3/11/2004|   1.7|     false|
|3/11/2004|   1.5|     false|
|3/11/2004|   1.6|     false|
|3/11/2004|   1.9|     false|
|3/11/2004|   2.9|      true|
|3/11/2004|   2.2|      true|
|3/11/2004|   2.2|      true|
|3/11/2004|   2.9|      true|
|3/11/2004|   4.8|      true|
|3/11/2004|   6.9|     false|
|3/11/2004|   6.1|     false|
|3/11/2004|   3.9|      true|
|3/11/2004|   1.5|     false|
|3/11/2004|   1.0|     false|
|3/12/2004

The output above confirms that the results match our expectations. The `check_numeric_range` method behaves correctly for a numeric range with both lower and upper bounds.

**Case 2** Provide lower bound only  
When only one bound is provided (lower = 2), the method applies only to the lower side. Missing data will produce a NULL value in the Boolean result column. Any value greater ≥ 2 returns true.

In [14]:
#case 2 provide one bound
check=sql_air_checker.check_numeric_range(column = 'CO(GT)', lower =2 , upper =None, new_column = 'new_CO(GT)')
check.select('Date', 'CO(GT)', 'new_CO(GT)').show(40)

+---------+------+----------+
|     Date|CO(GT)|new_CO(GT)|
+---------+------+----------+
|3/10/2004|   2.6|      true|
|3/10/2004|   2.0|      true|
|3/10/2004|   2.2|      true|
|3/10/2004|   2.2|      true|
|3/10/2004|   1.6|     false|
|3/10/2004|   1.2|     false|
|3/11/2004|   1.2|     false|
|3/11/2004|   1.0|     false|
|3/11/2004|   0.9|     false|
|3/11/2004|   0.6|     false|
|3/11/2004|  NULL|      NULL|
|3/11/2004|   0.7|     false|
|3/11/2004|   0.7|     false|
|3/11/2004|   1.1|     false|
|3/11/2004|   2.0|      true|
|3/11/2004|   2.2|      true|
|3/11/2004|   1.7|     false|
|3/11/2004|   1.5|     false|
|3/11/2004|   1.6|     false|
|3/11/2004|   1.9|     false|
|3/11/2004|   2.9|      true|
|3/11/2004|   2.2|      true|
|3/11/2004|   2.2|      true|
|3/11/2004|   2.9|      true|
|3/11/2004|   4.8|      true|
|3/11/2004|   6.9|      true|
|3/11/2004|   6.1|      true|
|3/11/2004|   3.9|      true|
|3/11/2004|   1.5|     false|
|3/11/2004|   1.0|     false|
|3/12/2004

The output above confirms that when only the lower bound is provided, the method evaluates the condition solely on that side. For rows with missing input values, the appended Boolean column correctly returns NULL, as intended.

**Case 3**: Provide only an upper bound (upper = 5)
When only the upper bound is supplied, the method applies the range check only to the upper side. Values ≤ 5 return true while values > 5 return false. Missing values produce NULL in the Boolean column.

In [16]:
#case 2 provide only an upper bound
check=sql_air_checker.check_numeric_range(column = 'CO(GT)', lower =None , upper =5, new_column = 'new')
check.select('Date', 'CO(GT)', 'new').show(40)

+---------+------+-----+
|     Date|CO(GT)|  new|
+---------+------+-----+
|3/10/2004|   2.6| true|
|3/10/2004|   2.0| true|
|3/10/2004|   2.2| true|
|3/10/2004|   2.2| true|
|3/10/2004|   1.6| true|
|3/10/2004|   1.2| true|
|3/11/2004|   1.2| true|
|3/11/2004|   1.0| true|
|3/11/2004|   0.9| true|
|3/11/2004|   0.6| true|
|3/11/2004|  NULL| NULL|
|3/11/2004|   0.7| true|
|3/11/2004|   0.7| true|
|3/11/2004|   1.1| true|
|3/11/2004|   2.0| true|
|3/11/2004|   2.2| true|
|3/11/2004|   1.7| true|
|3/11/2004|   1.5| true|
|3/11/2004|   1.6| true|
|3/11/2004|   1.9| true|
|3/11/2004|   2.9| true|
|3/11/2004|   2.2| true|
|3/11/2004|   2.2| true|
|3/11/2004|   2.9| true|
|3/11/2004|   4.8| true|
|3/11/2004|   6.9|false|
|3/11/2004|   6.1|false|
|3/11/2004|   3.9| true|
|3/11/2004|   1.5| true|
|3/11/2004|   1.0| true|
|3/12/2004|   1.7| true|
|3/12/2004|   1.9| true|
|3/12/2004|   1.4| true|
|3/12/2004|   0.8| true|
|3/12/2004|  NULL| NULL|
|3/12/2004|   0.6| true|
|3/12/2004|   0.8| true|


The output confirms that the method correctly evaluates the upper‑only condition and handles missing values as expected.

**Case 4**: No bound provided   
If no bounds are provided, a message is displayed reminding us that at least one bound is required, and the DataFrame is returned without modification.

In [12]:
# case 4: no bound provided
sql_air_checker.check_numeric_range(column = 'CO(GT)', lower = None, upper = None, new_column = 'new_CO(GT)').show(8)

Error: At least one of 'lower' or 'upper' must be provided.
+---+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|_c0|     Date|               Time|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|
+---+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|  0|3/10/2004|2026-03-17 18:00:00|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-17 19:00:00|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-17 20:00:00|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-17 21:00:00|   2.2| 1376|      80|     9.2|    948|    172|  1092|    122|  1584| 1203|11.0|60.0|0.7867|
|  4|3/10/2004|2026-03-17 22:00:00|   1.6| 1

26/03/17 17:49:56 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hzhou13@ncsu.edu/Project2/air.csv


The output above shows that this works as intended.

**Case 5**: provide a string column   
When a string column is supplied, the method issues a warning indicating that the selected column is not numeric. In this case, no validation is performed, and the method returns the original DataFrame without modification.

In [15]:
#case 5: provide a string colume
sql_air_checker.check_numeric_range(column = 'Date').show(8)

+---+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|_c0|     Date|               Time|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|
+---+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|  0|3/10/2004|2026-03-17 18:00:00|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-17 19:00:00|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-17 20:00:00|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-17 21:00:00|   2.2| 1376|      80|     9.2|    948|    172|  1092|    122|  1584| 1203|11.0|60.0|0.7867|
|  4|3/10/2004|2026-03-17 22:00:00|   1.6| 1272|      51|     6.5|    836|    131|  1205|    116|  1490|

26/03/17 18:13:09 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hzhou13@ncsu.edu/Project2/air.csv


The output above verifies that the method handles string columns correctly by preventing numeric validation and leaving the DataFrame unchanged.

Across the four test scenarios (1) providing both bounds, (2) providing a single lower bound, (3) providing a single upper bound, (4) providing no bounds, and (4) providing a string column, the method behaved exactly as expected. These results confirm that the check_numeric_range function is implemented correctly and handles all tested input conditions as intended.

### Testing `check_string_levels` method
Since the dataset does not contain any string variables with missing values, we create a binary variable `C6H6_label` from the numeric column `C6H6(GT)`. Benzene concentration (C6H6(GT)) is a key indicator of outdoor air quality. When the concentration is ≤ 5, the air quality is considered good; otherwise, it is considered poor. Therefore, we define C6H6(GT) ≤ 5 as `Good` and values greater than 5 as `Bad`.

In [22]:
#Step 1: Create a string/categorical column from numeric quality
#map to 'good' if <= 5 else 'bad'; preserve NaN as NaN
df_string = sql_air.withColumn(
    "C6H6_label",
    F.when(F.col("C6H6(GT)").isNull(), F.lit(None).cast(StringType()))
    .when(F.col("C6H6(GT)") <= 5, F.lit("Good"))
    .otherwise(F.lit("Bad"))
)
df_string.show(10)

+---+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+----------+
|_c0|     Date|               Time|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|C6H6_label|
+---+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+----------+
|  0|3/10/2004|2026-03-17 18:00:00|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|       Bad|
|  1|3/10/2004|2026-03-17 19:00:00|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|       Bad|
|  2|3/10/2004|2026-03-17 20:00:00|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|       Bad|
|  3|3/10/2004|2026-03-17 21:00:00|   2.2| 1376|      80|     9.2|    948|    172|  1092|    122|  1584| 1203|11.0|60.0|0.7867|       Bad|
|  4|3/10/2004|2026-03-17 2

26/03/17 18:47:42 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hzhou13@ncsu.edu/Project2/air.csv


In above table, I created a binary variable called `C6H6_label` based on the numeric variable `C6H6(GT)`. I defined concentrations ≤ 5 as good air quality, and values above 5 as bad air quality.

**Case 1**: The binary varibale `C6H6_label` is used to test `check_string_levels` method; We will check the levels = Good.   
In this test, the binary string column `C6H6_label` is used to evaluate the behavior of the `check_string_levels` method. We specify the expected level as `Good`, and the method will check whether each entry in the column matches this user‑defined set of levels.

In [19]:
# first create an instance of your SparkDataCheck class using your df DataFrame
df_string_checker = my_class.SparkDataCheck(df_string)

#case 1: check the levels = good
check = df_string_checker.check_string_levels(column = "C6H6_label", levels = ['Good'], new_column="is_good")
check.select('Date', 'C6H6(GT)', 'C6H6_label', 'is_good').show(10)
# Filter for rows where any of the specified columns are null and select them
check.filter(
    F.col("C6H6(GT)").isNull() | F.col("C6H6_label").isNull() | F.col("is_good").isNull()
).select('Date', 'C6H6(GT)', 'C6H6_label', 'is_good').show(10)

+---------+--------+----------+-------+
|     Date|C6H6(GT)|C6H6_label|is_good|
+---------+--------+----------+-------+
|3/10/2004|    11.9|       Bad|  false|
|3/10/2004|     9.4|       Bad|  false|
|3/10/2004|     9.0|       Bad|  false|
|3/10/2004|     9.2|       Bad|  false|
|3/10/2004|     6.5|       Bad|  false|
|3/10/2004|     4.7|      Good|   true|
|3/11/2004|     3.6|      Good|   true|
|3/11/2004|     3.3|      Good|   true|
|3/11/2004|     2.3|      Good|   true|
|3/11/2004|     1.7|      Good|   true|
+---------+--------+----------+-------+
only showing top 10 rows
+--------+--------+----------+-------+
|    Date|C6H6(GT)|C6H6_label|is_good|
+--------+--------+----------+-------+
|4/1/2004|    NULL|      NULL|   NULL|
|4/1/2004|    NULL|      NULL|   NULL|
|4/1/2004|    NULL|      NULL|   NULL|
|4/8/2004|    NULL|      NULL|   NULL|
|4/9/2004|    NULL|      NULL|   NULL|
|4/9/2004|    NULL|      NULL|   NULL|
|4/9/2004|    NULL|      NULL|   NULL|
|4/9/2004|    NULL|      

The results confirms that when each value in the string column `C6H6_label` is compared against the the user-specified set of levels (Good), the `check_string_levels` method behaves correctly. The method appends a Boolean column named `is_good`, where rows labeled *Good* return true and rows labeled *Bad* return false. Additionally, the results show that when `C6H6(GT)` is NULL, both `C6H6_label` and the derived `is_good` column are also NULL. Overall, the method performs as intended for this scenario.

**Case 2**: Check the levels = `Bad`  
In this case, we test the `check_string_levels` method by specifying `Bad` as the expected level. When `Bad` is provided as the target level, the method should return **true** for rows where C6H6_label is `Bad` and **false** for rows labeled `Good`. Missing values should produce **NULL** in the resulting Boolean column.

In [24]:
#case 2: cehck the levels = Bad
check = df_string_checker.check_string_levels(column = "C6H6_label", levels = ['Bad'], new_column="is_bad")
check.select('Date', 'C6H6(GT)', 'C6H6_label', 'is_bad').show(10)
# Filter for rows where any of the specified columns are null and select them
check.filter(
    F.col("C6H6(GT)").isNull() | F.col("C6H6_label").isNull() | F.col("is_bad").isNull()
).select('Date', 'C6H6(GT)', 'C6H6_label', 'is_bad').show(5)

+---------+--------+----------+------+
|     Date|C6H6(GT)|C6H6_label|is_bad|
+---------+--------+----------+------+
|3/10/2004|    11.9|       Bad|  true|
|3/10/2004|     9.4|       Bad|  true|
|3/10/2004|     9.0|       Bad|  true|
|3/10/2004|     9.2|       Bad|  true|
|3/10/2004|     6.5|       Bad|  true|
|3/10/2004|     4.7|      Good| false|
|3/11/2004|     3.6|      Good| false|
|3/11/2004|     3.3|      Good| false|
|3/11/2004|     2.3|      Good| false|
|3/11/2004|     1.7|      Good| false|
+---------+--------+----------+------+
only showing top 10 rows
+--------+--------+----------+------+
|    Date|C6H6(GT)|C6H6_label|is_bad|
+--------+--------+----------+------+
|4/1/2004|    NULL|      NULL|  NULL|
|4/1/2004|    NULL|      NULL|  NULL|
|4/1/2004|    NULL|      NULL|  NULL|
|4/8/2004|    NULL|      NULL|  NULL|
|4/9/2004|    NULL|      NULL|  NULL|
+--------+--------+----------+------+
only showing top 5 rows


The results indicate that when each value in the string column C6H6_label matches the user-specified level (`Bad`), the `check_string_levels` method correctly returns the DataFrame with an appended Boolean column `is_bad`. Values labeled `Bad` in the `C6H6_label` column return true in the `is_bad` column, while values labeled `Good` return false. The output also shows that when `C6H6(GT)` is NULL, both `C6H6_label` and the derived `is_bad` column are also NULL. Overall, the method is functioning as intended for this scenario.

**Case 3**: No string column is provided   
In this case, we intentionally select a column that is not of string type to test how the `check_string_levels` method handles invalid input. The method should detect that the column is not a string column, issue a warning, and return the original DataFrame without making any changes.

In [23]:
#case 3: no string column is provided
check = df_string_checker.check_string_levels(column = "S1_CO", levels = ['Good'], new_column="is_good")
check.show(5)

+---+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+----------+
|_c0|     Date|               Time|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|C6H6_label|
+---+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+----------+
|  0|3/10/2004|2026-03-17 18:00:00|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|       Bad|
|  1|3/10/2004|2026-03-17 19:00:00|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|       Bad|
|  2|3/10/2004|2026-03-17 20:00:00|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|       Bad|
|  3|3/10/2004|2026-03-17 21:00:00|   2.2| 1376|      80|     9.2|    948|    172|  1092|    122|  1584| 1203|11.0|60.0|0.7867|       Bad|
|  4|3/10/2004|2026-03-17 2

26/03/17 18:55:40 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hzhou13@ncsu.edu/Project2/air.csv


The results show that a warning is issued indicating that the selected column is not of string type, and the method returns the DataFrame without any modifications. The `check_string_levels` method behaves as intended for this scenario.

**Case 4**: No levels are provided (empty list)  
In this case, we intentionally supply an empty list of allowed levels to test how the `check_string_levels` method behaves when the user provides no valid levels for comparison. The method should warn that the list of levels is empty and then proceed with the default behavior.

In [25]:
#case 4: no levels are provided
check = df_string_checker.check_string_levels(column = "C6H6_label", levels = [], new_column="new")
check.select('Date', 'C6H6(GT)', 'C6H6_label', 'new').show(10)
# Filter for rows where any of the specified columns are null and select them
check.filter(
    F.col("C6H6(GT)").isNull() | F.col("C6H6_label").isNull() | F.col("new").isNull()
).select('Date', 'C6H6(GT)', 'C6H6_label', 'new').show(5)

+---------+--------+----------+-----+
|     Date|C6H6(GT)|C6H6_label|  new|
+---------+--------+----------+-----+
|3/10/2004|    11.9|       Bad|false|
|3/10/2004|     9.4|       Bad|false|
|3/10/2004|     9.0|       Bad|false|
|3/10/2004|     9.2|       Bad|false|
|3/10/2004|     6.5|       Bad|false|
|3/10/2004|     4.7|      Good|false|
|3/11/2004|     3.6|      Good|false|
|3/11/2004|     3.3|      Good|false|
|3/11/2004|     2.3|      Good|false|
|3/11/2004|     1.7|      Good|false|
+---------+--------+----------+-----+
only showing top 10 rows
+--------+--------+----------+----+
|    Date|C6H6(GT)|C6H6_label| new|
+--------+--------+----------+----+
|4/1/2004|    NULL|      NULL|NULL|
|4/1/2004|    NULL|      NULL|NULL|
|4/1/2004|    NULL|      NULL|NULL|
|4/8/2004|    NULL|      NULL|NULL|
|4/9/2004|    NULL|      NULL|NULL|
+--------+--------+----------+----+
only showing top 5 rows


The results show that when an empty list of levels is provided, the method issues a warning indicating that the levels argument is empty and returns the DataFrame with an appended column `new`. Regardless of whether the value in `C6H6_label` is `Good` or `Bad`, the new column correctly returns **false** for all non‑NULL entries. In addition, the output confirms that whenever `C6H6(GT)` is NULL, both `C6H6_label` and the derived `new` column are also **NULL**. Overall, the method behaves correctly for this scenario.

**Case 5**: Column is not in the dataset   
In this case, we intentionally provide an invalid (non‑existent) column name to test whether the `check_string_levels` method can correctly validate the input. When the specified column is not found in the DataFrame, the method should issue a warning and return the original DataFrame without making any modifications.

In [27]:
#Case 5: column is not in the dataset
check = df_string_checker.check_string_levels(column = "beneze", levels = ['Good'], new_column="new")
check.show(5)

Error: Column 'beneze' not found in DataFrame. Returning DataFrame without modification.
+---+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+----------+
|_c0|     Date|               Time|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|C6H6_label|
+---+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+----------+
|  0|3/10/2004|2026-03-17 18:00:00|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|       Bad|
|  1|3/10/2004|2026-03-17 19:00:00|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|       Bad|
|  2|3/10/2004|2026-03-17 20:00:00|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|       Bad|
|  3|3/10/2004|2026-03-17 21:00:00|   2.2| 1376|      80|     9.2|    948|   

26/03/17 19:11:16 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hzhou13@ncsu.edu/Project2/air.csv


The results indicate that when a column name that does not exist in the DataFrame is provided, the method prints a warning to inform the user that the variable is not present in the dataset. As expected, the method then returns the original DataFrame unchanged. This confirms that the `check_string_levels method` properly handles invalid column references.

Overall, the results across all five test cases confirm that the `check_string_levels` method is implemented correctly and behaves as expected under all tested conditions.

In [12]:
import pandas as pd
import numpy as np
import pyspark.pandas as ps
pdf = pd.read_csv("air.csv", delimiter = ",")
pdf.head()

,Unnamed: 0,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888
